# AI-Enhanced Correlation Workflow

§11.0.7 — This notebook demonstrates how to use machine learning
techniques to improve WeCo well correlation:

1. Autoencoder feature extraction from raw logs
2. Electrofacies clustering for facies-aware cost
3. Comparison: standard vs AI-enhanced correlation

In [ ]:
from weco.data import WellList
from weco.preprocessing import (
    compute_vshale, normalise_log, compute_electrofacies,
    autoencoder_features,
)
from weco.ext import ProjectExt
import matplotlib.pyplot as plt

In [ ]:
# Load wells
wl = WellList('data/data_set_shallow_marine/wells.txt')
print(f'Loaded {len(wl.wells)} wells')
for w in wl.wells:
    print(f'  {w.name}: {w.size} samples, logs: {list(w.data.keys())}')

## Step 1: Standard Conditioning

In [ ]:
for w in wl.wells:
    compute_vshale(w, gr_name='GR')
normalise_log(wl, 'GR', method='percentile')
print('Standard conditioning done')

## Step 2: AI Feature Extraction

In [ ]:
# Autoencoder features (requires PyTorch)
try:
    n_features = autoencoder_features(
        wl, log_names=['GR'], latent_dim=4, window_size=20, epochs=30,
    )
    print(f'Added {n_features} autoencoder features per well')
    
    # Visualize features for first well
    w = wl.wells[0]
    fig, axes = plt.subplots(1 + n_features, 1, figsize=(10, 3*(1+n_features)), sharex=True)
    axes[0].plot(w.data['Depth'], w.data['GR'])
    axes[0].set_ylabel('GR')
    for d in range(n_features):
        axes[1+d].plot(w.data['Depth'], w.data[f'AE_{d}'])
        axes[1+d].set_ylabel(f'AE_{d}')
    axes[-1].set_xlabel('Depth')
    plt.tight_layout()
    plt.show()
except ImportError as e:
    print(f'PyTorch not available: {e}')
    n_features = 0

In [ ]:
# Electrofacies clustering
try:
    compute_electrofacies(wl, log_names=['GR'], n_clusters=5)
    print('Electrofacies computed')
except ImportError as e:
    print(f'scikit-learn not available: {e}')

## Step 3: Compare Standard vs AI-Enhanced Correlation

In [ ]:
# Standard correlation
proj_std = ProjectExt()
proj_std.set_options_ext(cost_function='composite', var_data='GR', var_weight=1.0)
proj_std.run(wl)
res_std = proj_std.get_res_file()
print(f'Standard: {res_std.get_nbr_results()} results, best cost = {res_std.get_result_cost(0):.4f}')

# AI-enhanced (if features available)
if n_features > 0:
    proj_ai = ProjectExt()
    proj_ai.set_options_ext(
        cost_function='composite',
        var_data='AE_0',
        var_weight=1.0,
    )
    proj_ai.run(wl)
    res_ai = proj_ai.get_res_file()
    print(f'AI-enhanced: {res_ai.get_nbr_results()} results, best cost = {res_ai.get_result_cost(0):.4f}')

## Summary

AI-enhanced features capture **multi-scale patterns** that single-log
correlation misses. The autoencoder learns a compressed representation
that encodes both shape and context, while electrofacies clustering
provides discrete groupings for facies-aware cost functions.